# Test PaperQA with Three Locally-Hosted NIM APIs

Prerequisites:
- Launch NIMs -> `launch_NIMs.sh`
- Install PaperQA -> `install_PQA.sh`

***Use the Jupyter Kernel built in last step.***

In [ ]:
import logging
import sys

logging.basicConfig(level=logging.WARNING, format="%(levelname)s: %(message)s")
for _name in ("LiteLLM", "litellm"):
    _log = logging.getLogger(_name)
    _log.setLevel(logging.INFO)
    if not _log.handlers:
        _h = logging.StreamHandler(sys.stdout)
        _h.setLevel(logging.INFO)
        _h.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
        _log.addHandler(_h)
    _log.propagate = False

In [ ]:
# # Test if paperqa_nemotron is available
import paperqa
print(f"PaperQA location: {paperqa.__file__}")

# Log embedding call
from paperqa.llms import LiteLLMEmbeddingModel
_embed_log = logging.getLogger("LiteLLM")
_orig_embed = LiteLLMEmbeddingModel.embed_documents
async def _logged_embed(self, *args, **kwargs):
    texts = args[0] if args else kwargs.get("texts", [])
    n = len(texts) if texts else 0
    kind = "query" if n == 1 else "embed_documents"
    msg = f"LiteLLM embedding() model= {getattr(self, 'name', '?')}; ({kind}) n={n}"
    _embed_log.info(msg)
    if not _embed_log.handlers:
        print(f"INFO: {msg}")
    return await _orig_embed(self, *args, **kwargs)
LiteLLMEmbeddingModel.embed_documents = _logged_embed

try:
    import paperqa_nemotron
    from paperqa_nemotron import parse_pdf_to_pages
    print(f"✅ paperqa_nemotron location: {paperqa_nemotron.__file__}")
    print(f"✅ parse_pdf_to_pages available: {parse_pdf_to_pages}")
except ImportError as e:
    print(f"❌ paperqa_nemotron NOT installed!")
    print(f"   Error: {e}")
    print(f"   Run: pip install -e '.[local,pymupdf,nemotron]'")

## Step 1: Load PDF Papers (multiple)

In [ ]:
# Create papers directory
!mkdir -p papers

**Upload multiple PDFs to the `papers/` folder, then run the cell below to auto-detect all of them:**
I tested two PDFs: 
- [Attention Is All You Need](https://arxiv.org/abs/1706.03762)
- [Acid-sensing ion channels 1a (ASIC1a) inhibit neuromuscular transmission in female mice](https://pubmed.ncbi.nlm.nih.gov/24336653/)

In [ ]:
import os
import glob

# Auto-detect PDF files in the papers folder
pdf_files = sorted(glob.glob("papers/*.pdf"))

if pdf_files:
    print(f"📁 Found {len(pdf_files)} PDF(s) in papers/ (all will be added):")
    for i, pdf in enumerate(pdf_files):
        size_mb = os.path.getsize(pdf) / (1024 * 1024)
        print(f"   [{i}] {pdf} ({size_mb:.2f} MB)")
else:
    print("❌ No PDFs found in papers/ folder!")
    print("   Upload PDFs to papers/ and re-run this cell.")
    pdf_files = []

## Step 2: Configure Locally-Hosted Endpoints

We configure three locally-hosted NIMs (started via `launch_NIMs.sh`):

- **Parse** nvidia/nemotron-parse -> PDF parser (localhost:8002)
- **Embedding** nvidia/llama-3.2-nv-embedqa-1b-v2 -> embedding (localhost:8003)
- **VLM** nvidia/nemotron-nano-12b-v2-vl -> enrichment_llm, summary_llm, llm, agent_llm (localhost:8004)
  
*Right-side terms match paper-qa `Settings` / `ParsingSettings` field names.*

In [ ]:
# =============================================================================
# NIM CONFIGURATION (Parse=8002, Embedding=8003, VLM=8004)
# =============================================================================
import os
SELFHOST_API_KEY = "dummy"

# ----- Parse -----
PARSE_API_BASE = "http://localhost:8002/v1"
PARSE_API_KEY = "dummy"
PARSE_MODEL_NAME = "nvidia/nemotron-parse"
PARSE_COMPLETION_KWARGS = {
    "temperature": 0,
    "max_tokens": 8995,
}

# ----- Embedding --------------------
EMBEDDING_API_BASE = "http://localhost:8003/v1"
EMBEDDING_MODEL = "nvidia/llama-3.2-nv-embedqa-1b-v2"

# ----- VLM --------------------
VLM_API_BASE = "http://localhost:8004/v1"
VLM_MODEL = "nvidia/nemotron-nano-12b-v2-vl"
CUSTOM_VLM_NAME = "selfhost-nemotron-vlm"

print("\n=== Locally-Hosted NIM Endpoints ===")
print(f"Nemotron-Parse: {PARSE_API_BASE} ({PARSE_MODEL_NAME})")
print(f"Embedding: {EMBEDDING_API_BASE}  ({EMBEDDING_MODEL})")
print(f"VLM: {VLM_API_BASE}  ({VLM_MODEL})")

## Step 3: Create PaperQA Settings

In [ ]:
from paperqa import Settings
from paperqa.settings import AgentSettings, IndexSettings, AnswerSettings, ParsingSettings
from paperqa_nemotron import parse_pdf_to_pages
import pathlib

# VLM config for enrichment_llm, summary_llm, llm, agent_llm (locally-hosted nemotron-nano-12b-v2-vl on 8004)
nvidia_vlm_config = {
    "model_list": [
        {
            "model_name": CUSTOM_VLM_NAME,
            "litellm_params": {
                "model": f"openai/{VLM_MODEL}",
                "api_base": VLM_API_BASE,
                "api_key": SELFHOST_API_KEY,
                "temperature": 0,
                "max_tokens": 2048,
            },
        }
    ]
}

# Embedding config (locally-hosted llama-3.2-nv-embedqa-1b-v2 on 8003)
nvidia_embedding_config = {
    "kwargs": {
        "api_base": EMBEDDING_API_BASE,
        "api_key": SELFHOST_API_KEY,
        "encoding_format": "float",
        "input_type": "passage",     
    }
}

# ParsingSettings with Nemotron-Parse NIM
parsing_settings = ParsingSettings(
    # Use nemotron-parse as the PDF parser
    parse_pdf=parse_pdf_to_pages,
    
    # reader_config is passed to parse_pdf_to_pages(); api_params go to LiteLLM
    reader_config={
        "chunk_chars": 5000,
        "overlap": 250,
        "dpi": 150,  # Image resolution for PDF rendering
        
        "api_params": {
            "api_base": PARSE_API_BASE,       
            "api_key": PARSE_API_KEY,         
            "model_name": PARSE_MODEL_NAME, 
            **PARSE_COMPLETION_KWARGS, 
        }
    },
    
    # Enrichment LLM for multimodal (self-hosted VLM on 8004)
    enrichment_llm=CUSTOM_VLM_NAME,
    enrichment_llm_config=nvidia_vlm_config,
    multimodal=True,  # Enable multimodal to use nemotron-parse's full capabilities
)

# Full settings
settings = Settings(
    # LLMs
    llm=CUSTOM_VLM_NAME,
    llm_config=nvidia_vlm_config,
    summary_llm=CUSTOM_VLM_NAME,
    summary_llm_config=nvidia_vlm_config,
    
    # Embedding (self-hosted on 8003)
    embedding=f"openai/{EMBEDDING_MODEL}",
    embedding_config=nvidia_embedding_config,
    
    # Temperature for all LLMs (answer, summary, agent, enrichment) unless overridden in their config
    temperature=0,
    # Global logging level 0-3 for LLM/embedding calls (not per-model)
    verbosity=3,
    
    answer=AnswerSettings(
        evidence_k=5,
        answer_max_sources=3,
    ),
    
    # Use our nemotron-parse settings
    parsing=parsing_settings,
    
    agent=AgentSettings(
        agent_llm=CUSTOM_VLM_NAME,
        agent_llm_config=nvidia_vlm_config,
        index=IndexSettings(
            paper_directory=pathlib.Path.cwd() / "papers",
        ),
    ),
)

print("✅ PaperQA Settings created!")
print(f"   PDF Parser: {parsing_settings.parse_pdf}")
print(f"   Nemotron Parse API Base: {parsing_settings.reader_config['api_params']['api_base']}")
print(f"   Multimodal: {parsing_settings.multimodal}")
print(f"   Embedding: {settings.embedding}")
print(f"   LLM: {settings.llm}")

## Step 4: Add All Papers with Nemotron-Parse

In [ ]:
from paperqa import Docs

# Create a Docs object and add all PDFs
docs = Docs()

if not pdf_files:
    print("❌ No PDFs to add. Run the cell above to detect PDFs in papers/.")
else:
    print(f"Adding {len(pdf_files)} paper(s) to Docs (using nemotron-parse)...")
    added = 0
    for i, path in enumerate(pdf_files):
        try:
            name = await docs.aadd(path, settings=settings)
            if name:
                added += 1
                print(f"   [{i+1}/{len(pdf_files)}] ✅ {path}")
            else:
                print(f"   [{i+1}/{len(pdf_files)}] ⚠️ Skipped (already in collection): {path}")
        except Exception as e:
            print(f"   [{i+1}/{len(pdf_files)}] ❌ Failed: {path}")
            print(f"      Error: {e}")
            import traceback
            traceback.print_exc()
    print(f"\n✅ Added {added} doc(s). Total docs: {len(docs.docs)}")
    for doc_key, doc in docs.docs.items():
        print(f"   - {doc.docname}: {doc.citation[:80]}...")

## Step 5: `docs.aquery()` -> non-agent, no tools

In [ ]:
# Query the docs
print("Querying docs...")

try:
    session = await docs.aquery(
        "What experiments are carried out?",
        settings=settings,
    )
    
    print("✅ Query SUCCESS!")
    print("\n" + "=" * 60)
    print("Question:")
    print("=" * 60)
    print(session.question)
    
    print("\n" + "=" * 60)
    print("Answer:")
    print("=" * 60)
    print(session.answer)
    
    print("\n" + "=" * 60)
    print("References:")
    print("=" * 60)
    print(session.references)
    
except Exception as e:
    print(f"❌ Query FAILED!")
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

## Step 6 `agent_query()` -> agent, tools


In [ ]:
from paperqa.agents.main import agent_query

agent_question = "What experiments are carried out?"
print(f"Running agent_query with docs (pre-loaded papers)...")
print(f"Question: {agent_question}\n")

try:
    response = await agent_query(agent_question, settings, docs=docs)
    print("✅ agent_query SUCCESS!")
    print(f"Status: {response.status}")
    print("\n" + "=" * 60)
    print("Answer:")
    print("=" * 60)
    print(response.session.answer or "(no answer)")
except Exception as e:
    print(f"❌ agent_query failed: {e}")
    import traceback
    traceback.print_exc()

## Step 6.5: `ask()` -> agent, tools

Test the full agent workflow:

In [ ]:
from paperqa import ask

print("Running agent-based query with ask()...")
print("This will search, gather evidence, and generate an answer.\n")

try:
    response = await ask(
        "What experiments are carried out?",
        settings=settings,
    )
    
    print("✅ Agent query SUCCESS!")
    print(f"\n--- Status: {response.status} ---")
    
    print("\n" + "=" * 60)
    print("Question:")
    print("=" * 60)
    print(response.session.question)
    
    print("\n" + "=" * 60)
    print("Answer:")
    print("=" * 60)
    print(response.session.answer)
    
    print("\n" + "=" * 60)
    print("Contexts Used (top 3):")
    print("=" * 60)
    for i, ctx in enumerate(response.session.contexts[:3], 1):
        print(f"\nContext {i} (score: {ctx.score}):")
        print(f"  Source: {ctx.text.name}")
        print(f"  Summary: {ctx.context[:200]}...")
        
except Exception as e:
    print(f"❌ Agent query FAILED!")
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()